# 🦠 Exploratory Data Analysis on COVID-19 Global Data

## Overview
This project analyzes the global spread of COVID-19 to uncover trends in confirmed cases, deaths, recoveries, and vaccination progress across countries.

**Dataset:** [COVID-19 Dataset – Kaggle](https://www.kaggle.com/datasets/imdevskp/corona-virus-report)  
**File:** `covid_19_clean_complete.csv`

## Importing all the Important Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully ✅')

## Loading the Dataset

In [ ]:
df = pd.read_csv('covid_19_clean_complete.csv')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns ✅')

## Checking First Five Rows of Dataset

In [ ]:
df.head()

## Checking Last Five Rows of Dataset

In [ ]:
df.tail()

## Total Number of Rows and Columns Present in Dataset

In [ ]:
df.shape

## Checking Data Types of Each Column

In [ ]:
df.dtypes

## An Overview on Dataset

In [ ]:
df.info()

# Performing Data Cleaning

## Checking Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
# % of missing values per column
(df.isnull().sum() / df.shape[0]) * 100

## Handling Missing Values

In [ ]:
# Fill numeric missing values with 0 (no reported cases = 0)
for col in ['Confirmed', 'Deaths', 'Recovered', 'Active']:
    if col in df.columns:
        df[col].fillna(0, inplace=True)

# Fill Province/State missing with 'National'
if 'Province/State' in df.columns:
    df['Province/State'].fillna('National', inplace=True)

print('Missing values handled ✅')
df.isnull().sum()

## Checking for Duplicates

In [ ]:
df.duplicated().sum()

## Data Type Conversion

In [ ]:
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract month and year
df['Month'] = df['Date'].dt.month_name()
df['Year']  = df['Date'].dt.year
df['Week']  = df['Date'].dt.isocalendar().week

print('Date converted ✅')
df[['Date','Month','Year']].head()

## Compute Mortality and Recovery Rates

In [ ]:
# Add derived columns
df['Mortality_Rate']  = np.where(df['Confirmed'] > 0,
                                  (df['Deaths'] / df['Confirmed']) * 100, 0)
df['Recovery_Rate']   = np.where(df['Confirmed'] > 0,
                                  (df['Recovered'] / df['Confirmed']) * 100, 0)

print('Derived metrics computed ✅')
df[['Confirmed','Deaths','Recovered','Mortality_Rate','Recovery_Rate']].describe()

## Statistical Summary

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
# Correlation heatmap
numcols = ['Confirmed','Deaths','Recovered','Active'] if 'Active' in df.columns \
           else ['Confirmed','Deaths','Recovered']
plt.figure(figsize=(7, 5))
sns.heatmap(df[numcols].corr(), annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap — COVID-19 Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Performing Exploratory Data Analysis

## i) Univariate Analysis

In [ ]:
# --- Global Daily Confirmed Cases Over Time ---
global_daily = df.groupby('Date')[['Confirmed','Deaths','Recovered']].sum().reset_index()

plt.figure(figsize=(13, 5))
plt.plot(global_daily['Date'], global_daily['Confirmed'], color='#E63946', lw=2, label='Confirmed')
plt.fill_between(global_daily['Date'], global_daily['Confirmed'], alpha=0.1, color='#E63946')
plt.xlabel('Date')
plt.ylabel('Total Cases')
plt.title('Global COVID-19 Confirmed Cases Over Time', fontsize=13, fontweight='bold')
plt.legend()
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Exponential growth in cases visible from March 2020')

In [ ]:
# --- Global Deaths vs Recoveries Over Time ---
plt.figure(figsize=(13, 5))
plt.plot(global_daily['Date'], global_daily['Deaths'],    color='#E63946', lw=2, label='Deaths')
plt.plot(global_daily['Date'], global_daily['Recovered'], color='#2A9D8F', lw=2, label='Recovered')
plt.xlabel('Date')
plt.ylabel('Cumulative Count')
plt.title('Global Deaths vs Recoveries Over Time', fontsize=13, fontweight='bold')
plt.legend()
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Recoveries consistently outpace deaths globally')

In [ ]:
# --- Top 10 Countries by Total Confirmed Cases ---
country_col = 'Country/Region' if 'Country/Region' in df.columns else 'Country'
latest_date = df['Date'].max()
latest = df[df['Date'] == latest_date].groupby(country_col)['Confirmed'].sum()
top10 = latest.nlargest(10)

plt.figure(figsize=(10, 5))
bars = plt.barh(top10.index[::-1], top10.values[::-1], color='#E63946')
plt.xlabel('Total Confirmed Cases')
plt.title('Top 10 Countries — Total Confirmed Cases', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top10.values[::-1]):
    plt.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
             f'{val:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## ii) Bivariate Analysis

In [ ]:
# --- Top 10 Countries by Deaths ---
top10_deaths = df[df['Date'] == latest_date].groupby(country_col)['Deaths'].sum().nlargest(10)

plt.figure(figsize=(10, 5))
bars = plt.barh(top10_deaths.index[::-1], top10_deaths.values[::-1], color='#457B9D')
plt.xlabel('Total Deaths')
plt.title('Top 10 Countries — Total Deaths', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top10_deaths.values[::-1]):
    plt.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
             f'{val:,.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Mortality Rate: Top 15 Countries (min 1000 cases) ---
mort = df[df['Date'] == latest_date].groupby(country_col).agg(
    Confirmed=('Confirmed','sum'), Deaths=('Deaths','sum')).reset_index()
mort = mort[mort['Confirmed'] >= 1000]
mort['MortalityRate'] = (mort['Deaths'] / mort['Confirmed']) * 100
top_mort = mort.nlargest(15, 'MortalityRate')

plt.figure(figsize=(10, 6))
sns.barplot(x='MortalityRate', y=country_col, data=top_mort, palette='Reds_r')
plt.xlabel('Mortality Rate (%)')
plt.title('Top 15 Countries by Mortality Rate (min 1K cases)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Monthly New Cases (Global) ---
monthly = df.groupby(['Year','Month'])['Confirmed'].sum().reset_index()
# Simple year+month sort
monthly['period'] = monthly['Year'].astype(str) + '-' + monthly['Month'].str[:3]
plt.figure(figsize=(13, 5))
plt.bar(range(len(monthly)), monthly['Confirmed'], color='#E63946', edgecolor='white')
plt.xticks(range(len(monthly)), monthly['period'], rotation=60, ha='right', fontsize=7)
plt.ylabel('Confirmed Cases')
plt.title('Monthly Cumulative Confirmed Cases (Global)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## iii) Multivariate Analysis

In [ ]:
# --- Confirmed vs Deaths Scatter (Top 20 countries) ---
top20 = df[df['Date'] == latest_date].groupby(country_col).agg(
    Confirmed=('Confirmed','sum'), Deaths=('Deaths','sum')).nlargest(20,'Confirmed').reset_index()

plt.figure(figsize=(10, 6))
plt.scatter(top20['Confirmed'], top20['Deaths'], s=80, color='#E63946', edgecolors='white', linewidth=0.5)
for _, row in top20.iterrows():
    plt.annotate(row[country_col], (row['Confirmed'], row['Deaths']),
                 textcoords='offset points', xytext=(5, 3), fontsize=7)
plt.xlabel('Total Confirmed Cases')
plt.ylabel('Total Deaths')
plt.title('Confirmed Cases vs Deaths — Top 20 Countries', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Recovery Rate: Top 10 Countries ---
recov = df[df['Date'] == latest_date].groupby(country_col).agg(
    Confirmed=('Confirmed','sum'), Recovered=('Recovered','sum')).reset_index()
recov = recov[recov['Confirmed'] >= 10000]
recov['RecoveryRate'] = (recov['Recovered'] / recov['Confirmed']) * 100
top_recov = recov.nlargest(10, 'RecoveryRate')

plt.figure(figsize=(10, 5))
sns.barplot(x='RecoveryRate', y=country_col, data=top_recov, palette='Greens_r')
plt.xlabel('Recovery Rate (%)')
plt.title('Top 10 Countries by Recovery Rate (min 10K cases)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Global 3-Metric Trend ---
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
metrics   = ['Confirmed', 'Deaths', 'Recovered']
colors    = ['#E63946', '#457B9D', '#2A9D8F']
for ax, metric, color in zip(axes, metrics, colors):
    ax.plot(global_daily['Date'], global_daily[metric], color=color, lw=2)
    ax.fill_between(global_daily['Date'], global_daily[metric], alpha=0.12, color=color)
    ax.set_ylabel(metric, fontsize=10)
    ax.set_title(f'Global {metric} Trend', fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.suptitle('Global COVID-19 Trends Overview', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Key Insights & Findings

### 1. Case Growth
- COVID-19 showed **exponential growth** from March 2020 onwards.
- Multiple waves visible in the time series data.

### 2. Top Affected Countries
- **USA, India, and Brazil** consistently rank in top 3 for confirmed cases.

### 3. Mortality Rate
- Ranges from <0.5% to >10% across countries — reflects healthcare capacity.

### 4. Recovery
- Global recovery rate improved significantly after mid-2020.

### 5. Seasonal Pattern
- Cases tend to surge in **winter months** in Northern Hemisphere countries.

## Conclusion
This EDA of the COVID-19 global dataset highlights the unprecedented scale of the pandemic, with exponential growth, unequal country-level impacts, and improving recovery rates over time.

---
### 📌 Next Steps
- Build a **time-series forecasting model** for case prediction.
- Analyze the **impact of vaccination** on case reduction.
- Create an **interactive dashboard** with Plotly/Dash.

**📊 Tools Used:** Python, Pandas, Matplotlib, Seaborn, Jupyter Notebook

---
### -----------------------------------------THANKYOU---------------------------------